## Import Libraries

In [1]:
import sys
import os

sys.path.append(os.path.abspath('..'))

In [2]:
from dotenv import load_dotenv
from pathlib import Path
from scipy.optimize import brentq
from scipy.interpolate import interp1d
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import roc_curve
from qdrant_client import QdrantClient
from qdrant_client import models
from torch.utils.data import DataLoader, TensorDataset
from torch.nn import CrossEntropyLoss
from torch.optim import Adam
from utils.embedding_model import embedding_model
import numpy as np
import time
import torch
import psutil
from tqdm import tqdm
import wandb

## Setup Training Variables

In [3]:
model_name = 'embedding_v2'
ratio = '80:10:10'
train_split = '80'
seeder = 42
window_len = os.getenv("WINDOW_SIZE")
stride_len = os.getenv("STRIDE")
num_batch = os.getenv("BATCH_SIZE")
num_epoch = os.getenv("EPOCHS")
margin = 0.1
wandb_name = model_name + '_train_' + train_split + '_' + str(seeder) + '_' + str(window_len) + '_' + str(stride_len) + '_b' + str(num_batch) + '_e' + str(num_epoch) + '_margin_' + str(margin)
print(wandb_name)

embedding_v2_train_80_42_1000_500_b16_e100_margin_0.1


In [4]:
load_dotenv(".env")
BASE_PATH = os.getenv("BASE_PATH")
PREPROCESSED_PATH = os.getenv("PREPROCESSED_PATH")
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

eo_df = np.load(Path(BASE_PATH +PREPROCESSED_PATH) / 'X_eo_normalized.npy')
ec_df = np.load(Path(BASE_PATH +PREPROCESSED_PATH) / 'X_ec_normalized.npy')
y_eo_data = np.load(Path(BASE_PATH +PREPROCESSED_PATH) / 'y_eo_data.npy')
y_ec_data = np.load(Path(BASE_PATH +PREPROCESSED_PATH) / 'y_ec_data.npy')


In [6]:
wandb.login(key=os.getenv("WANDB_API_KEY"))
run = wandb.init(
    entity="chocomaltt",
    project="eeg-biometric-system",
    name=wandb_name,
    config={
        "model_name": model_name,
        "ratio": ratio,
        "train_split": train_split,
        "seeder": seeder,
        "window_len": os.getenv("WINDOW_SIZE"),
        "stride_len": os.getenv("STRIDE"),
        "num_batch": os.getenv("BATCH_SIZE"),
        "epoch": os.getenv("EPOCHS")
    },
    tags=[model_name, 'train_' + str(train_split), str(seeder), str(window_len), str(stride_len), str(num_batch), str(num_epoch), str(margin)]
)

process = psutil.Process(os.getpid())
initial_memory = psutil.virtual_memory()
wandb.log({
    "resource/logging_check": 1,
    "resource/cpu_percent": psutil.cpu_percent(interval=1),
    "resource/process_cpu_percent": process.cpu_percent(interval=None),
    "resource/memory_percent": initial_memory.percent,
    "resource/memory_used_gb": initial_memory.used / (1024 ** 3),
    "resource/process_memory_gb": process.memory_info().rss / (1024 ** 3),
})

wandb: Loading settings from /home/chocomaltt/.config/wandb/settings
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for http://localhost:8080.
wandb: Appending key for localhost:8080 to your netrc file: /home/chocomaltt/.netrc
wandb: Currently logged in as: chocomaltt to http://localhost:8080. Use `wandb login --relogin` to force relogin


In [7]:
eo_labels = list(y_eo_data)
eo_labels = np.array(eo_labels)

print("shape eo_df: ", eo_df.shape)
print("labels: ", eo_labels)

# First split: 70% train, 30% temporary pool for validation/test.
sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, temp_idx = next(sss1.split(eo_df, eo_labels))

X_train = eo_df[train_idx]
y_train = eo_labels[train_idx]

X_temp = eo_df[temp_idx]
y_temp = eo_labels[temp_idx]

# Second split: split the remaining 30% equally into 15% validation and 15% test.
sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
val_idx_temp, test_idx_temp = next(sss2.split(X_temp, y_temp))

X_val = X_temp[val_idx_temp]
X_test = X_temp[test_idx_temp]

y_val = y_temp[val_idx_temp]
y_test = y_temp[test_idx_temp]


shape eo_df:  (6431, 64, 320)
labels:  [  0   0   0 ... 108 108 108]


In [8]:
X_train_t = torch.from_numpy(X_train.copy()).float()
y_train_t = torch.from_numpy(y_train.copy()).long()

X_test_t = torch.from_numpy(X_test.copy()).float()
y_test_t = torch.from_numpy(y_test.copy()).long()

X_val_t = torch.from_numpy(X_val.copy()).float()
y_val_t = torch.from_numpy(y_val.copy()).long()

train_ds = TensorDataset(X_train_t, y_train_t)
val_ds = TensorDataset(X_val_t, y_val_t)
test_ds = TensorDataset(X_test_t, y_test_t)

train_loader = DataLoader(
    train_ds,
    batch_size=int(os.getenv("BATCH_SIZE")),
    shuffle=True,
    num_workers=int(os.getenv("NUM_WORKERS")),
    drop_last=True
)
val_loader = DataLoader(
    val_ds,
    batch_size=int(os.getenv("BATCH_SIZE")),
    shuffle=False,
    num_workers=int(os.getenv("NUM_WORKERS")),
    drop_last=False
)
test_loader = DataLoader(
    test_ds,
    batch_size=int(os.getenv("BATCH_SIZE")),
    shuffle=False,
    num_workers=int(os.getenv("NUM_WORKERS")),
    drop_last=False
)

In [9]:
model = embedding_model(
    in_channels=int(os.getenv("INPUT_CHANNELS")),
    num_classes=int(os.getenv("NUM_CLASSES")),
)
model.to(os.getenv("DEVICE"))

embedding_model(
  (input): Sequential(
    (0): LazyConv2d(0, 64, kernel_size=(1, 1), stride=(1, 1), padding=same)
    (1): SELU()
  )
  (conv2_temporal): Sequential(
    (0): LazyConv2d(0, 32, kernel_size=(4, 4), stride=(1, 1), padding=same)
    (1): SELU()
  )
  (batch_normalization): LazyBatchNorm2d(0, eps=32, momentum=0.1, affine=True, track_running_stats=True)
  (elu): ELU(alpha=1.0)
  (MaxPool2d): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
  (conv2_spatial): Sequential(
    (0): LazyConv2d(0, 64, kernel_size=(2, 2), stride=(1, 1), padding=same)
    (1): SELU()
  )
  (lstm): LSTM(2048, 128, batch_first=True)
  (dense): Sequential(
    (0): LazyLinear(in_features=0, out_features=128, bias=True)
    (1): SELU()
  )
)

In [10]:
# Pastikan DEVICE sudah di-set (GPU kalau ada, kalau nggak CPU)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Lakukan "Dry Run" untuk membangunkan layer Lazy
with torch.no_grad():
    # Ambil 1 sampel saja dari X_train_t (Ingat, ECG sudah kita buang)
    sample_eeg = X_train_t[:1].to(DEVICE, non_blocking=True)

    sample_eeg = sample_eeg.unsqueeze(1)
    
    # Masukkan ke model. Setelah baris ini lewat, dimensi layer Lazy resmi terbentuk!
    _ = model(sample_eeg)

# 3. Hitung Parameter
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✓ Model berjalan di: {DEVICE}")
print(f"✓ Model initialized - Total params: {total_params:,}, Trainable: {trainable_params:,}")

# 4. Cek Memori GPU (Opsional)
if torch.cuda.is_available():
    print(f"GPU Memory: {torch.cuda.memory_allocated()/1e9:.2f}GB allocated")

✓ Model berjalan di: cuda
✓ Model initialized - Total params: 1,172,896, Trainable: 1,172,896
GPU Memory: 0.01GB allocated


/home/chocomaltt/Kuliah/eeg-biometric-system/eeg/lib/python3.10/site-packages/torch/nn/modules/conv.py:548: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1025.)
  return F.conv2d(


In [11]:
client = QdrantClient(url="http://localhost:6333")

if not client.collection_exists("eeg_embeddings_v2"):
    client.create_collection(
        collection_name="eeg_embeddings_v2",
        vectors_config=models.VectorParams(size=128, distance=models.Distance.COSINE),
    )

In [12]:
from pytorch_metric_learning import losses # Import library metric learning

LEARNING_RATE = float(os.getenv("LEARNING_RATE", 1e-4))
EPOCHS = int(os.getenv("EPOCHS", 100))

# 1. Ganti Loss Function menjadi Triplet Margin Loss
# Margin 0.2 adalah standar yang bagus untuk permulaan
criterion = losses.TripletMarginLoss(margin=margin)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

checkpoint_filepath = "best_eeg_embedding_model.pth"
best_val_loss = float('inf')
best_epoch = 0
patience = 1000 # Kesabaran bisa dinaikkan sedikit untuk metric learning
wait = 0
best_weights = None

# History (Kita hilangkan akurasi sementara, karena akurasi embedding 
# dihitung secara terpisah nanti menggunakan KNN/Cosine Similarity)
history = {'loss': [], 'val_loss': []}

process = psutil.Process(os.getpid())
psutil.cpu_percent(interval=None)
process.cpu_percent(interval=None)

print(f"Starting Embedding Training with Early Stopping (patience={patience})...")

for epoch in range(EPOCHS):
    # --- TRAINING PHASE ---
    model.train()
    train_loss = 0.0
    
    for batch_idx, (data_eeg, targets) in enumerate(train_loader):
        if data_eeg.dim() == 3: 
            data_eeg = data_eeg.unsqueeze(1)
        data_eeg = data_eeg.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        
        optimizer.zero_grad()

        # print(f"train batch shape: {data_eeg.shape}")
        #
        # Outputnya sekarang adalah VEKTOR EMBEDDING
        embeddings = model(data_eeg) 
        
        # Triplet loss akan otomatis mencari pasangan (Anchor, Positive, Negative)
        # berdasarkan label (targets) yang kamu berikan
        loss = criterion(embeddings, targets)
        
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    avg_train_loss = train_loss / len(train_loader)

    # --- VALIDATION PHASE ---
    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for data_eeg, targets in val_loader:
            data_eeg = data_eeg.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)
            
            embeddings = model(data_eeg)
            loss = criterion(embeddings, targets)
            
            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)

    # Simpan History
    history['loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)

    memory = psutil.virtual_memory()
    process_memory = process.memory_info().rss / (1024 ** 3)

    wandb.log({
        "epoch/epoch": epoch,
        "epoch/train_loss": avg_train_loss,
        "epoch/val_loss": avg_val_loss,
        "epoch/best_val_loss": best_val_loss,
        "epoch/best_epoch": best_epoch,
        "epoch/patience": patience,
        "epoch/wait": wait,
        "epoch/best_weights": best_weights,
        "epoch/checkpoint_filepath": checkpoint_filepath,
        "epoch/optimizer_state_dict": optimizer.state_dict(),
        "resource/cpu_percent": psutil.cpu_percent(interval=None),
        "resource/process_cpu_percent": process.cpu_percent(interval=None),
        "resource/memory_percent": memory.percent,
        "resource/memory_used_gb": memory.used / (1024 ** 3),
        "resource/process_memory_gb": process_memory,
    })

    print(f"Epoch {epoch+1:03d}/{EPOCHS} | Train Loss (Triplet): {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    # --- CHECKPOINT & EARLY STOPPING ---
    if avg_val_loss < best_val_loss:
        print(f" -> Validation loss improved ({best_val_loss:.4f} to {avg_val_loss:.4f}). Saving model... 💾")
        best_val_loss = avg_val_loss
        best_epoch = epoch
        best_weights = model.state_dict().copy()
        wait = 0
        
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val_loss': best_val_loss,
        }, checkpoint_filepath)
    else:
        wait += 1
        
    if wait >= patience:
        print(f"\nEarly stopping triggered! No improvement for {patience} epochs.")
        if best_weights is not None:
            model.load_state_dict(best_weights)
            print(f"Restored best model weights from Epoch {best_epoch+1}.")
        break

# After full training without early stop, last epoch may not be best — always use best checkpoint
if best_weights is not None:
    model.load_state_dict(best_weights)

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Training finished. Vektor biometrik siap digunakan! 🚀")

Starting Embedding Training with Early Stopping (patience=1000)...


wandb: WARNING Artifact "source-eeg-biometric-system-_home_chocomaltt_Kuliah_eeg-biometric-system_notebooks_02_model_training.ipynb" already exists with the same content. No new version will be created.


Epoch 001/100 | Train Loss (Triplet): 0.0688 | Val Loss: 0.0505
 -> Validation loss improved (inf to 0.0505). Saving model... 💾
Epoch 002/100 | Train Loss (Triplet): 0.0736 | Val Loss: 0.1397
Epoch 003/100 | Train Loss (Triplet): 0.0707 | Val Loss: 0.0452
 -> Validation loss improved (0.0505 to 0.0452). Saving model... 💾
Epoch 004/100 | Train Loss (Triplet): 0.0561 | Val Loss: 0.0553
Epoch 005/100 | Train Loss (Triplet): 0.0645 | Val Loss: 0.0707
Epoch 006/100 | Train Loss (Triplet): 0.0668 | Val Loss: 0.0469
Epoch 007/100 | Train Loss (Triplet): 0.0608 | Val Loss: 0.0585
Epoch 008/100 | Train Loss (Triplet): 0.0610 | Val Loss: 0.0537
Epoch 009/100 | Train Loss (Triplet): 0.0605 | Val Loss: 0.0470
Epoch 010/100 | Train Loss (Triplet): 0.0609 | Val Loss: 0.0467
Epoch 011/100 | Train Loss (Triplet): 0.0619 | Val Loss: 0.0493
Epoch 012/100 | Train Loss (Triplet): 0.0535 | Val Loss: 0.0440
 -> Validation loss improved (0.0452 to 0.0440). Saving model... 💾
Epoch 013/100 | Train Loss (Triple

In [13]:
# Enrollment gallery: train + val (test held out for evaluation)
model.eval()
X_enroll = np.concatenate([X_train, X_val], axis=0)
y_enroll = np.concatenate([y_train, y_val], axis=0)

emb_batch = int(os.getenv("BATCH_SIZE"))
enroll_ds = TensorDataset(
    torch.from_numpy(X_enroll).float(),
    torch.from_numpy(y_enroll).long(),
)
enroll_loader = DataLoader(
    enroll_ds,
    batch_size=emb_batch,
    shuffle=False,
    num_workers=int(os.getenv("NUM_WORKERS")),
    drop_last=False,
)

emb_chunks, label_chunks = [], []
with torch.no_grad():
    for data_eeg, targets in tqdm(enroll_loader, desc="Extract embeddings (enrollment)"):
        data_eeg = data_eeg.to(DEVICE, non_blocking=True)
        emb = model(data_eeg).cpu().numpy()
        emb_chunks.append(emb)
        label_chunks.append(targets.numpy())

embeddings_matrix = np.concatenate(emb_chunks, axis=0)
subject_ids = np.concatenate(label_chunks, axis=0)

out_path = Path(BASE_PATH + PREPROCESSED_PATH) / "embeddings_eo_train_val.npz"
out_path.parent.mkdir(parents=True, exist_ok=True)
np.savez_compressed(out_path, embeddings=embeddings_matrix, subject_ids=subject_ids)
print(f"Saved local embedding backup: {out_path}  shape={embeddings_matrix.shape}")

qdrant_batch = 256
for start in tqdm(
    range(0, len(embeddings_matrix), qdrant_batch),
    desc="Upsert to Qdrant",
):
    end = min(start + qdrant_batch, len(embeddings_matrix))
    points = [
        models.PointStruct(
            id=start + i,
            vector=embeddings_matrix[start + i].tolist(),
            payload={"subject_id": int(subject_ids[start + i])},
        )
        for i in range(end - start)
    ]
    client.upsert(collection_name="eeg_embeddings_v2", points=points)

print(f"Upserted {len(embeddings_matrix)} points to collection 'eeg_embeddings_v2'.")

Extract embeddings (enrollment): 100%|██████████| 362/362 [00:05<00:00, 71.73it/s]


Saved local embedding backup: Dataset/preprocessed/embeddings_eo_train_val.npz  shape=(5787, 128)


Upsert to Qdrant: 100%|██████████| 23/23 [00:00<00:00, 28.10it/s]

Upserted 5787 points to collection 'eeg_embeddings_v2'.


# model = torch.load('../best_eeg_embedding_model.pth')
coba modifikasi arsitektur model (conv 1d -> 2d)
perkecil kernel size (8 -> ...)
cek performance per satu subject (waktu, accuracy)
cek usage cpu + memory

In [14]:
# client = QdrantClient("/home/chocomaltt/Kuliah/eeg-biometric-system/qdrant_storage/collections/eeg_embeddings")
client = QdrantClient(url="http://localhost:6333")
model.eval()

y_true = []
y_scores = []

with torch.no_grad():
    for data_eeg, targets in test_loader:
        data_eeg = data_eeg.to(DEVICE, non_blocking=True)

        embeddings = model(data_eeg).cpu().numpy()
        targets = targets.numpy()

        for i in range(len(embeddings)):
            query_vector = embeddings[i].tolist()
            true_label = targets[i]

            search_result = client.query_points(
                collection_name="eeg_embeddings_v2",
                query=query_vector,
                limit=1
            )
            # print(search_result)

            if search_result:
                best_match = search_result.points[0]
                similarity_score = best_match.score
                predicted_label = best_match.payload["subject_id"]

                is_genuine = 1 if (predicted_label == true_label) else 0

                y_true.append(is_genuine)
                y_scores.append(similarity_score)

y_true = np.array(y_true)
y_scores = np.array(y_scores)

fpr, tpr, thresholds = roc_curve(y_true, y_scores)
eer_threshold = brentq(lambda x: 1.0 - x - interp1d(fpr, tpr)(x), 0.0, 1.0)

far = fpr
frr = 1 - tpr

eer = interp1d(far, frr)(eer_threshold)
top1_accuracy = sum(y_true) / len(y_true)

memory = psutil.virtual_memory()
process = psutil.Process(os.getpid())
wandb.log({
    "eval/top1_accuracy": top1_accuracy,
    "eval/top1_accuracy_percent": top1_accuracy * 100,
    "eval/eer": float(eer),
    "eval/eer_percent": float(eer) * 100,
    "eval/eer_threshold": float(eer_threshold),
    "eval/total_test_samples": len(y_true),
    "resource/cpu_percent": psutil.cpu_percent(interval=None),
    "resource/process_cpu_percent": process.cpu_percent(interval=None),
    "resource/memory_percent": memory.percent,
    "resource/memory_used_gb": memory.used / (1024 ** 3),
    "resource/process_memory_gb": process.memory_info().rss / (1024 ** 3),
})

print("\n=== HASIL EVALUASI BIOMETRIK ===")
print(f"Total Sampel Test : {len(y_true)}")
print(f"Akurasi Top-1     : {top1_accuracy * 100:.2f}%")
print(f"EER (Makin kecil makin bagus) : {eer * 100:.2f}%")
print(f"Threshold Ideal   : {eer_threshold:.4f}")

wandb.finish()


=== HASIL EVALUASI BIOMETRIK ===
Total Sampel Test : 644
Akurasi Top-1     : 99.22%
EER (Makin kecil makin bagus) : 23.00%
Threshold Ideal   : 0.2300


epoch/best_epoch,▁▁▁▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇████
epoch/best_val_loss,█▇▇▇▇▇▆▆▆▅▄▄▄▃▃▃▃▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇█████
epoch/patience,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/train_loss,█▆▇▇▇▇▇▆▆▅▅▅▄▄▃▃▃▃▃▂▃▃▂▃▃▂▂▁▂▁▂▁▂▁▁▁▂▁▁▁
epoch/val_loss,▆▆█▅▇▅▆▆▄▄▃▃▃▂▃▃▂▂▃▂▂▂▂▂▂▂▂▂▂▁▁▂▁▂▁▁▂▂▂▁
epoch/wait,▁▂▃▄▆▂▃▁▃▃▁▂▁▂▃▄▅▆█▁▅▂▃▁▃▃▃▅▇▂▅▇▂▁▃▅▇▁▂▇
eval/eer,▁
eval/eer_percent,▁
eval/eer_threshold,▁
+9,...
